In [ ]:
%pip install pgmpy -q

In [ ]:
import itertools
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pgmpy.factors.discrete import DiscreteFactor
from pgmpy.inference import VariableElimination

# in versiunile noi de pgmpy clasa se numeste DiscreteMarkovNetwork, in cele vechi MarkovNetwork
try:
    from pgmpy.models import DiscreteMarkovNetwork as ReteMarkov
except ImportError:
    from pgmpy.models import MarkovNetwork as ReteMarkov

**Ex. 1**

**a)** $A \perp C \mid B$ inseamna ca, odata ce stim valoarea lui $B$, valoarea lui $C$ nu ne mai spune nimic in plus despre $A$:

$$P(a, c \mid b) = P(a \mid b)\,P(c \mid b) \iff P(a \mid b, c) = P(a \mid b)$$

Fara sa stim $B$, variabilele $A$ si $C$ pot fi dependente, pentru ca informatia circula prin $B$. Cand $B$ este cunoscut, el blocheaza acest drum, deci $C$ nu aduce nicio informatie suplimentara despre $A$.

**b)** Presupunem $A \perp C \mid B$. Din regula produsului si independenta conditionata:

$$P(a, b, c) = P(a, c \mid b)\,P(b) = P(a \mid b)\,P(c \mid b)\,P(b)$$

Alegem $\phi_{AB}(a, b) = P(a \mid b)$, $\phi_{BC}(b, c) = P(c \mid b)\,P(b)$ si $Z = 1$. Cum distributia comuna este strict pozitiva, toate marginalele si conditionatele sunt strict pozitive, deci $\phi_{AB}, \phi_{BC} > 0$ si $Z > 0$, iar

$$P(a, b, c) = \frac{1}{Z}\,\phi_{AB}(a, b)\,\phi_{BC}(b, c).$$

**c)** Presupunem ca $P(a, b, c) = \frac1Z \phi_{AB}(a, b)\,\phi_{BC}(b, c)$. Notam $f(b) = \sum_a \phi_{AB}(a, b)$ si $g(b) = \sum_c \phi_{BC}(b, c)$. Atunci

$$P(b) = \sum_{a, c} P(a, b, c) = \frac{1}{Z}\, f(b)\, g(b)$$

$$P(a, c \mid b) = \frac{P(a, b, c)}{P(b)} = \frac{\phi_{AB}(a, b)\,\phi_{BC}(b, c)}{f(b)\, g(b)}$$

Sumand dupa $c$, respectiv dupa $a$, obtinem

$$P(a \mid b) = \frac{\phi_{AB}(a, b)}{f(b)}, \qquad P(c \mid b) = \frac{\phi_{BC}(b, c)}{g(b)}$$

Deci $P(a \mid b)\,P(c \mid b) = \frac{\phi_{AB}(a, b)\,\phi_{BC}(b, c)}{f(b)\, g(b)} = P(a, c \mid b)$, adica $A \perp C \mid B$. $\blacksquare$

In [ ]:
# ex. 2 a): graful ales (un arbore cu n = 5 statii) si vecinii fiecarei statii
stari = ["Soare", "Ploaie", "Nori"]
statii = [f"S{i}" for i in range(1, 6)]
muchii = [("S1", "S2"), ("S2", "S3"), ("S2", "S4"), ("S4", "S5")]

G = nx.Graph(muchii)
for s in statii:
    print(s, "are vecinii", list(G.neighbors(s)))

# un graf conex cu n noduri si n - 1 muchii, fara cicluri, este arbore
print("e arbore (fara cicluri):", nx.is_tree(G))

nx.draw(G, with_labels=True, node_size=1500, node_color="lightblue")
plt.show()

In [ ]:
# ex. 2 b): modelul markov cu factori unari si pe perechi
# probabilitatile initiale (aceleasi pentru toate statiile), in ordinea Soare, Ploaie, Nori
p = np.array([0.5, 0.3, 0.2])

# potentialul pe muchii: randurile si coloanele sunt in ordinea Soare, Ploaie, Nori
# nu e o matrice de probabilitati conditionate, e doar un potential
Psi = np.array([[0.7, 0.2, 0.1],
                [0.2, 0.6, 0.2],
                [0.1, 0.3, 0.6]])

model = ReteMarkov()
model.add_nodes_from(statii)
model.add_edges_from(muchii)

# factorul unar phi_i(s) = p_i(s) pentru fiecare statie
factori = [DiscreteFactor([s], [3], p, state_names={s: stari}) for s in statii]

# factorul pe perechi phi_ij(s_i, s_j) = Psi[s_i, s_j]; ordinea variabilelor e cea din tuplul muchiei
# (prima variabila = randurile lui Psi, a doua = coloanele)
factori += [DiscreteFactor([u, v], [3, 3], Psi, state_names={u: stari, v: stari}) for u, v in muchii]

model.add_factors(*factori)
assert model.check_model()

# ex. 2 c): graful modelului
nx.draw(nx.Graph(model.edges()), with_labels=True, node_size=1500, node_color="lightgreen")
plt.show()

**Ex. 2 c)** Distributia comuna, cu muchiile $E = \{(1,2), (2,3), (2,4), (4,5)\}$:

$$P(s_1, \dots, s_5) = \frac{1}{Z} \prod_{i=1}^{5} \phi_i(s_i) \prod_{(i,j) \in E} \phi_{ij}(s_i, s_j), \qquad \phi_i(s) = p_i(s), \quad \phi_{ij}(s_i, s_j) = \Psi_{s_i, s_j}$$

unde $Z$ este constanta de normalizare, adica suma produsului peste toate cele $3^5 = 243$ de configuratii.

In [ ]:
# ex. 2 d): observam S1 = Soare si S5 = Nori, apoi calculam marginalele statiilor neobservate
infer = VariableElimination(model)
obs = {"S1": "Soare", "S5": "Nori"}
neobs = [s for s in statii if s not in obs]

marginale = {}
for s in neobs:
    q = infer.query([s], evidence=obs)
    q.normalize()   # pgmpy poate intoarce valori nenormalizate pe retele markov
    marginale[s] = q
    print(s, dict(zip(stari, q.values.round(4))))
    print("   MAP marginal:", stari[int(np.argmax(q.values))])

# verificare prin enumerarea tuturor celor 3^5 configuratii compatibile cu observatiile
def scor(cfg):
    # produsul factorilor unari si pe perechi pentru o configuratie (indici 0, 1, 2)
    r = np.prod([p[cfg[s]] for s in statii])
    for u, v in muchii:
        r *= Psi[cfg[u], cfg[v]]
    return r

scoruri = {}
for c in itertools.product(range(3), repeat=5):
    cfg = dict(zip(statii, c))
    if all(stari[cfg[k]] == v for k, v in obs.items()):
        scoruri[c] = scor(cfg)
Z = sum(scoruri.values())

for k, s in enumerate(statii):
    if s in neobs:
        m = np.zeros(3)
        for c, sc in scoruri.items():
            m[c[k]] += sc / Z
        print(s, "(enumerare):", m.round(4))

In [ ]:
# ex. 2 e): configuratia comuna MAP a statiilor neobservate, dat observatiile
map_comun = infer.map_query(neobs, evidence=obs)
print("MAP comun:", map_comun)

# alegerea separata a maximelor marginale
map_marginal = {s: stari[int(np.argmax(marginale[s].values))] for s in neobs}
print("MAP marginal pe fiecare statie:", map_marginal)
print("coincid:", map_comun == map_marginal)

# verificare cu enumerarea: configuratia cu scorul cel mai mare si probabilitatea ei
cel_mai_bun = max(scoruri, key=scoruri.get)
print("enumerare:", [stari[i] for i in cel_mai_bun], "cu probabilitatea", round(scoruri[cel_mai_bun] / Z, 4))

**Interpretare ex. 2.** Cu $S_1 = Soare$ si $S_5 = Nori$ observate, marginalele sunt $S_2$: (0.83, 0.13, 0.05), $S_3$: (0.72, 0.20, 0.08) si $S_4$: (0.51, 0.24, 0.25) pe ordinea Soare, Ploaie, Nori. Statiile vecine cu $S_1$ si cele apropiate de ea tind sa aiba tot Soare, iar incertitudinea creste spre $S_4$, aflata langa statia cu Nori. Configuratia comuna MAP este (Soare, Soare, Soare) pentru $S_2, S_3, S_4$, cu probabilitatea $0.40$, si coincide aici cu alegerea separata a maximelor marginale. In general cele doua **nu coincid intotdeauna**: maximul marginal alege cea mai probabila stare a fiecarei statii luate separat, iar MAP-ul comun alege cea mai probabila configuratie intreaga, iar aceste doua criterii pot da raspunsuri diferite.

In [ ]:
# ex. 3 a): imaginea originala 5x5, un patrat alb (1) 3x3 pe fond negru (0)
x_orig = np.zeros((5, 5), dtype=int)
x_orig[1:4, 1:4] = 1
print(x_orig)

# ex. 3 b): inversam aproximativ 10% dintre pixeli, cu samanta fixa pentru reproductibilitate
rng = np.random.default_rng(23)
zgomot = rng.random((5, 5)) < 0.1
y = np.where(zgomot, 1 - x_orig, x_orig)
print("pixeli modificati:", zgomot.sum())
print(y)

fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(x_orig, cmap="gray", vmin=0, vmax=1); ax[0].set_title("originala")
ax[1].imshow(y, cmap="gray", vmin=0, vmax=1);      ax[1].set_title("observata (y)")
plt.show()

**Ex. 3 c)** Primul termen din energie, $\lambda \sum_i (x_i - y_i)^2$, penalizeaza diferenta dintre fiecare pixel $x_i$ si valoarea observata $y_i$ (**fidelitate fata de observatii**). Al doilea termen, $\sum_{(i,j) \in E} (x_i - x_j)^2$, penalizeaza vecinii care au valori diferite (**netezime**, ideea ca pixelii vecini tind sa fie la fel). Parametrul $\lambda > 0$ stabileste cat de mult conteaza primul termen fata de al doilea. Potentialele sunt

$$\phi_i(x_i) = e^{-\lambda (x_i - y_i)^2}, \qquad \phi_{ij}(x_i, x_j) = e^{-(x_i - x_j)^2}$$

iar produsul lor este $\prod_i \phi_i \prod_{(i,j)} \phi_{ij} = e^{-E(x; y)} \propto P(x \mid y)$, deci a minimiza energia inseamna a maximiza probabilitatea (configuratia MAP).

In [ ]:
# ex. 3 c): reteaua markov pe grila 5x5, cu vecini directi (sus, jos, stanga, dreapta)
def nume(i, j):
    return f"x{i}{j}"

def construieste_mrf(y, lmbda):
    model = ReteMarkov()
    model.add_nodes_from([nume(i, j) for i in range(5) for j in range(5)])
    factori = []
    for i in range(5):
        for j in range(5):
            # factor unar: exp(-lambda * (x - y_ij)^2) pentru x = 0 si x = 1
            factori.append(DiscreteFactor([nume(i, j)], [2],
                                          [np.exp(-lmbda * (v - y[i, j]) ** 2) for v in (0, 1)]))
            # muchii spre dreapta si in jos (ca fiecare muchie sa apara o singura data)
            for di, dj in [(0, 1), (1, 0)]:
                a, b = i + di, j + dj
                if a < 5 and b < 5:
                    model.add_edge(nume(i, j), nume(a, b))
                    # factor pe perechi: exp(-(xi - xj)^2) pentru cele 4 combinatii
                    factori.append(DiscreteFactor([nume(i, j), nume(a, b)], [2, 2],
                                                  [np.exp(-(u - v) ** 2) for u in (0, 1) for v in (0, 1)]))
    model.add_factors(*factori)
    return model

# graful grilei (acelasi pentru orice lambda)
m0 = construieste_mrf(y, 1.0)
assert m0.check_model()
pozitii = {nume(i, j): (j, -i) for i in range(5) for j in range(5)}
nx.draw(nx.Graph(m0.edges()), pozitii, with_labels=True, node_size=500, node_color="lightyellow", font_size=7)
plt.show()

In [ ]:
# ex. 3 d) si e): configuratia MAP exacta (variable elimination) pentru mai multe valori ale lui lambda
# folosim inferenta exacta, nu ICM, deci obtinem minimul global al energiei
def energie(x, y, lmbda):
    e = lmbda * np.sum((x - y) ** 2)
    e += np.sum((x[:, :-1] - x[:, 1:]) ** 2) + np.sum((x[:-1, :] - x[1:, :]) ** 2)
    return e

valori_lambda = [0.3, 1, 1.8, 3]
reconstructii = {}
for lmbda in valori_lambda:
    mrf = construieste_mrf(y, lmbda)
    map_x = VariableElimination(mrf).map_query(list(mrf.nodes()))
    x_rec = np.array([[map_x[nume(i, j)] for j in range(5)] for i in range(5)])
    reconstructii[lmbda] = x_rec
    print(f"lambda = {lmbda}: pixeli gresiti fata de original = {(x_rec != x_orig).sum()}, "
          f"energie = {energie(x_rec, y, lmbda):.2f}")

print("observata y: pixeli gresiti fata de original =", (y != x_orig).sum())

fig, ax = plt.subplots(1, 2 + len(valori_lambda), figsize=(14, 3))
ax[0].imshow(x_orig, cmap="gray", vmin=0, vmax=1); ax[0].set_title("originala")
ax[1].imshow(y, cmap="gray", vmin=0, vmax=1);      ax[1].set_title("observata")
for k, lmbda in enumerate(valori_lambda):
    ax[2 + k].imshow(reconstructii[lmbda], cmap="gray", vmin=0, vmax=1)
    ax[2 + k].set_title(f"lambda = {lmbda}")
plt.show()

**Interpretare ex. 3.** Imaginea observata are 2 pixeli gresiti: o gaura in centrul patratului si un pixel alb izolat in coltul din stanga jos. Reconstructiile exacte (MAP) pentru valorile lui $\lambda$ sunt:

- $\lambda = 0.3$ si $\lambda = 1$: imaginea iese complet neagra, cu 9 pixeli gresiti. Termenul de netezime domina, iar patratul este "netezit" pana dispare.
- $\lambda = 1.8$: imaginea reconstruita coincide cu originala, deci 0 pixeli gresiti. Gaura este umpluta si pixelul izolat este eliminat, iar patratul se pastreaza.
- $\lambda = 3$: gaura este umpluta, dar pixelul izolat din colt ramane, deci 1 pixel gresit. Fidelitatea fata de observatii incepe sa domine, iar pentru $\lambda$ si mai mare reconstructia devine identica cu observatia.

Un $\lambda$ mic favorizeaza netezimea (elimina zgomotul, dar poate sterge si detalii reale), iar un $\lambda$ mare favorizeaza fidelitatea fata de observatii (pastreaza detaliile, dar si zgomotul). Pentru ca am folosit inferenta exacta, fiecare imagine este minimul global al energiei, nu un minim local ca la ICM. Atentie: energia $E(x; y)$ nu este probabilitate, ci $P(x \mid y) \propto e^{-E(x; y)}$.